# 03 · 多重检验与基因组控制 Multiple Testing & Genomic Control

> **能力标签**：GB2（关联测试与表型 / Association Testing & Phenotype）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释 **多重假设检验问题**（multiple testing problem）及其在 GWAS 中的严重性。
2. 陈述**全基因组显著阈值**（genome-wide significance threshold）$p < 5 \times 10^{-8}$ 的来源（Bonferroni 校正）。
3. 绘制并解读 **QQ 图**（quantile-quantile plot），区分零假设偏差与真实信号。
4. 计算**基因组膨胀因子**（genomic inflation factor）$\lambda_{\text{GC}}$，识别系统性偏差来源。
5. 理解并实现**基于秩的逆正态变换**（rank-based inverse normal transform, INT），用于稳健化偏斜表型。

> 对应能力：**GB2**


## 概念讲解 Concepts

### 1 · 多重检验问题 Multiple Testing Problem

GWAS 同时检验 $m \approx 10^6$ 个 SNP，若每个测试的显著水平（significance level）为 $\alpha = 0.05$，
则在零假设下期望出现约 50,000 个假阳性（false positives）。

**Bonferroni 校正（Bonferroni correction）**：把单次测试的阈值降低到

$$\alpha_{\text{Bonferroni}} = \frac{\alpha}{m}$$

取 $\alpha = 0.05$，$m = 10^6$ 个独立 SNP：$\alpha_{\text{Bonf}} = 5 \times 10^{-8}$。

这就是 GWAS 领域约定俗成的**全基因组显著阈值（genome-wide significance threshold）**。

> 实际 SNP 并非独立（存在 LD），$m_{\text{eff}} < m$，故 $5 \times 10^{-8}$ 是保守但被广泛接受的阈值。

---

### 2 · QQ 图 Quantile-Quantile Plot

**QQ 图**把观测 $p$ 值的分布与零假设（uniform $[0,1]$）期望分布对比：
- x 轴：期望 $-\log_{10}(p)$，即 $-\log_{10}\left(\frac{k}{n+1}\right)$（第 $k$ 小的期望分位数）
- y 轴：观测 $-\log_{10}(p)$（升序排列）

**理想零假设下**：点应落在对角线 $y = x$ 附近。

**偏离模式的解读**：
- 整体上移（inflation）：群体分层、系统误差 -> 需要基因组控制
- 尾部突出：真实遗传信号 -> 科学发现

---

### 3 · 基因组膨胀因子 Genomic Inflation Factor $\lambda_{\text{GC}}$

$\lambda_{\text{GC}}$ 用于量化检验统计量是否系统性偏大（inflation）：

$$\lambda_{\text{GC}} = \frac{\text{median}(\chi^2_{\text{obs}})}{\text{median}(\chi^2_{1,0.5})}$$

其中 $\chi^2_{\text{obs}} = t^2$（$t$ 统计量的平方，等价于 $\chi^2_1$），
$\chi^2_{1,0.5} \approx 0.4549$（$\chi^2_1$ 分布的中位数）。

| $\lambda_{\text{GC}}$ | 解释 |
|-----------------------|------|
| $\approx 1.0$ | 统计量分布与零假设一致 |
| $> 1.05$ | 存在膨胀，可能有群体分层或技术偏差 |
| $< 1.0$ | 统计量保守（罕见） |

---

### 4 · 逆正态变换 Inverse Normal Transform (INT)

实际研究中，数量性状（如 BMI、血液指标）常呈现**偏斜分布（skewed distribution）**。
线性模型假设残差正态，偏斜会导致检验偏差。

**基于秩的 INT（rank-based inverse normal transform）**：

$$\text{INT}(x_i) = \Phi^{-1}\!\left(\frac{r_i - c}{n - 2c + 1}\right)$$

其中 $r_i$ 是 $x_i$ 的秩（rank），$c = 0.5$（偏移量，避免 $\Phi^{-1}(0)$ 或 $\Phi^{-1}(1)$），
$\Phi^{-1}$ 是标准正态分布的分位数函数（probit function）。

变换后的 $\text{INT}(x)$ 近似服从 $\mathcal{N}(0, 1)$，改善线性模型的假设满足度。


## 示例 Worked Example

演示多重检验、QQ 图、基因组膨胀因子 $\lambda_{\text{GC}}$ 和逆正态变换 INT。

In [ ]:
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile, os

rng = np.random.default_rng(2024)
tmpdir = tempfile.mkdtemp()

# 合成 p 值
m = 10000

# 情景 A：纯零假设（所有 SNP 无关联）
pvals_null = rng.uniform(0, 1, m)

# 情景 B：含真实信号 (20 个 causal SNP)
n_causal = 20
pvals_signal = pvals_null.copy()
pvals_signal[:n_causal] = stats.chi2.sf(
    rng.noncentral_chisquare(df=1, nonc=15, size=n_causal), df=1)

# 情景 C：有群体分层膨胀——把 chi2 统计量乘以 1.15
chi2_inflated = stats.chi2.isf(pvals_null, df=1) * 1.15
pvals_inflated = stats.chi2.sf(chi2_inflated, df=1)

print(f"零假设 p 值 均值：{pvals_null.mean():.4f}  (理论 0.5)")
print(f"信号场景 min-p：{pvals_signal.min():.2e}")
print(f"膨胀场景 min-p：{pvals_inflated.min():.2e}")


In [ ]:
# 实现 genomic_lambda
def genomic_lambda(pvals):
    """基因组膨胀因子 lambda_GC = median(观测 chi2) / median(chi2_1df 理论中位数)."""
    p = np.asarray(pvals, dtype=float)
    chi2_obs = stats.chi2.isf(p, df=1)
    chi2_med_expected = stats.chi2.ppf(0.5, df=1)  # ≈ 0.4549
    return float(np.median(chi2_obs) / chi2_med_expected)

# 实现 qq_points
def qq_points(pvals):
    """QQ 图数据：返回 (期望 -log10p, 观测 -log10p)，均升序。"""
    p = np.sort(np.asarray(pvals, dtype=float))
    n = len(p)
    expected = -np.log10(np.arange(1, n + 1) / (n + 1))
    observed = -np.log10(p)
    return expected[::-1], observed[::-1]

lam_null     = genomic_lambda(pvals_null)
lam_signal   = genomic_lambda(pvals_signal)
lam_inflated = genomic_lambda(pvals_inflated)

print(f"lambda_GC (零假设场景)   = {lam_null:.4f}  (期望 ~ 1.00)")
print(f"lambda_GC (含真实信号)   = {lam_signal:.4f}  (接近 1，尾部信号不影响中位数)")
print(f"lambda_GC (群体分层膨胀) = {lam_inflated:.4f}  (显著 > 1)")
print()
print(f"chi2_1 分布中位数 (理论) = {stats.chi2.ppf(0.5, 1):.6f}")


In [ ]:
# 绘制 QQ 图
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

scenarios = [
    (pvals_null,     f"零假设 (lambda={lam_null:.3f})",       "steelblue"),
    (pvals_signal,   f"含真实信号 (lambda={lam_signal:.3f})", "seagreen"),
    (pvals_inflated, f"群体分层膨胀 (lambda={lam_inflated:.3f})", "tomato"),
]

for ax, (pv, title, color) in zip(axes, scenarios):
    exp_lp, obs_lp = qq_points(pv)
    step = max(1, len(exp_lp) // 500)
    ax.scatter(exp_lp[::step], obs_lp[::step], s=8, alpha=0.6, color=color)
    lim = max(exp_lp.max(), obs_lp.max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", linewidth=0.8, label="y=x (期望)")
    ax.set_xlim(0, exp_lp.max() * 1.05)
    ax.set_ylim(0, obs_lp.max() * 1.05)
    ax.set_xlabel("期望 -log10(p)")
    ax.set_ylabel("观测 -log10(p)")
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=8)

fig.suptitle("QQ 图比较（三种场景）", fontsize=11)
fig.tight_layout()
out = os.path.join(tmpdir, "qq_plots.png")
fig.savefig(out, dpi=120)
plt.close(fig)
print(f"QQ 图已保存：{out}")


In [ ]:
# 多重检验：Bonferroni 阈值示例
alpha = 0.05
n_tests = m
bonf_threshold = alpha / n_tests
gwas_threshold = 5e-8

print(f"检验数量 m = {n_tests}")
print(f"Bonferroni 阈值 = {alpha} / {n_tests} = {bonf_threshold:.2e}")
print(f"GWAS 全基因组显著阈值 = {gwas_threshold:.0e}  (m=10^6 时的 Bonferroni)")
print()
print("零假设场景（应无信号）：")
print(f"  p < {bonf_threshold:.2e} 的 SNP 数 = {(pvals_null < bonf_threshold).sum()}")
print()
print("真实信号场景：")
print(f"  p < {bonf_threshold:.2e} 的 SNP 数 = {(pvals_signal < bonf_threshold).sum()}")
print(f"  其中前 {n_causal} 个是真正的 causal SNP")
print()
print(f"期望假阳性数（零假设下）= alpha * m = {alpha} * {n_tests} = {alpha * n_tests:.0f}")
print("Bonferroni 控制后期望假阳性 < 1")


In [ ]:
# 实现逆正态变换 INT
def rank_int(x, offset=0.5):
    """基于秩的逆正态变换（rank-based inverse normal transform）。
    把任意分布的 x 近似变成标准正态。offset=0.5 是常用偏移量。"""
    x = np.asarray(x, dtype=float)
    ranks = stats.rankdata(x)
    n = len(x)
    q = (ranks - offset) / (n - 2 * offset + 1)
    return stats.norm.ppf(q)

# 演示：对偏斜表型做 INT
n_demo = 1000
y_skewed = rng.lognormal(mean=3.0, sigma=0.5, size=n_demo)
y_int = rank_int(y_skewed)

print(f"原始表型（log-normal）：均值={y_skewed.mean():.2f}  偏度={stats.skew(y_skewed):.3f}")
print(f"INT 变换后：          均值={y_int.mean():.4f}  偏度={stats.skew(y_int):.4f}")
print(f"INT 变换后标准差 = {y_int.std():.4f}  (接近 1)")

sw_orig = stats.shapiro(y_skewed[:50])
sw_int  = stats.shapiro(y_int[:50])
print("Shapiro-Wilk 检验（样本 n=50）：")
print(f"  原始表型 p = {sw_orig.pvalue:.4f}  (p << 0.05 -> 拒绝正态)")
print(f"  INT 变换后 p = {sw_int.pvalue:.4f}  (p > 0.05 -> 不拒绝正态)")


In [ ]:
# 可视化 INT 变换效果
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(y_skewed, bins=40, color="tomato", edgecolor="white", linewidth=0.3)
axes[0].set_title("原始表型（对数正态分布）")
axes[0].set_xlabel("值")
axes[0].set_ylabel("频数")

axes[1].hist(y_int, bins=40, color="steelblue", edgecolor="white", linewidth=0.3)
axes[1].set_title("INT 变换后（近似标准正态）")
axes[1].set_xlabel("值")

(osm, osr), (slope, intercept, r) = stats.probplot(y_int, dist="norm")
axes[2].scatter(osm, osr, s=5, alpha=0.5, color="steelblue")
axes[2].plot([-4, 4], [slope * (-4) + intercept, slope * 4 + intercept],
             "r--", linewidth=1, label=f"R2={r**2:.4f}")
axes[2].set_xlabel("理论正态分位数")
axes[2].set_ylabel("样本分位数")
axes[2].set_title("INT 后 QQ 图（vs 标准正态）")
axes[2].legend(fontsize=8)

fig.tight_layout()
out = os.path.join(tmpdir, "int_demo.png")
fig.savefig(out, dpi=120)
plt.close(fig)
print(f"INT 演示图已保存：{out}")


In [ ]:
# 数值验证：lambda_GC 在纯零假设下应接近 1
n_rep = 200
lambdas = []
for _ in range(n_rep):
    pv = rng.uniform(0, 1, 5000)
    lambdas.append(genomic_lambda(pv))

lambdas = np.array(lambdas)
print(f"lambda_GC（纯零假设，{n_rep} 次重复，m=5000）：")
print(f"  均值   = {lambdas.mean():.4f}  (期望 1.000)")
print(f"  标准差 = {lambdas.std():.4f}")
print(f"  95% CI = [{np.percentile(lambdas, 2.5):.4f}, {np.percentile(lambdas, 97.5):.4f}]")
print("验证：均值接近 1，区间包含 1")

chi2_med = stats.chi2.ppf(0.5, 1)
print(f"chi2_1 分布中位数（理论） = {chi2_med:.6f}")
print(f"chi2_1 分布中位数（数值验证）= "
      f"{np.median(stats.chi2.isf(rng.uniform(0,1,100000), df=1)):.6f}")


## 动手 Exercise

以下模拟数据 `pvals_ex` 来自混合场景（真实信号 + 群体分层）。请你：

1. **计算** $\lambda_{\text{GC}}$，判断是否存在膨胀。
2. **绘制** QQ 图（可只打印期望和观测 $-\log_{10}(p)$ 的前 10 个点作为验证）。
3. 假设 $m = 500{,}000$ 个 SNP，计算 Bonferroni 校正后的显著阈值，统计 `pvals_ex` 中超过该阈值的 SNP 数。

```python
rng_ex = np.random.default_rng(77)
m_ex = 8000
chi2_base = rng_ex.chisquare(df=1, size=m_ex)
chi2_inflated_ex = chi2_base * 1.08
pvals_ex = stats.chi2.sf(chi2_inflated_ex, df=1)
pvals_ex[:5] = stats.chi2.sf(rng_ex.noncentral_chisquare(1, nonc=20, size=5), df=1)

# 你的代码写在这里
```

另外，对以下偏斜表型 `y_ex` 应用 `rank_int`，打印变换前后的偏度（skewness）：

```python
y_ex = rng_ex.exponential(scale=2.0, size=500)
# 你的代码写在这里
```


## 延伸阅读 Further Reading

1. **Devlin & Roeder (1999)**. "Genomic control for association studies." *Biometrics.* $\lambda_{\text{GC}}$ 的原始论文。
2. **Dudbridge & Gusnanto (2008)**. "Estimation of significance thresholds for genomewide association scans." *Genetic Epidemiology.* 讨论 $5 \times 10^{-8}$ 阈值的统计基础。
3. **Blom (1958)**. "Statistical estimates and transformed beta-variables." INT 偏移量的理论来源（本课用 $c = 0.5$ 的 van der Waerden 变体）。
4. **Price et al. (2010)**. "New approaches to population stratification in genome-wide association studies." *Nature Reviews Genetics.*


## 关联练习 Related Assignments

### b7-gc：基因组控制

完成 `progress/<你>/work/b7-gc/solution.py`，实现 `genomic_lambda(pvals)` 和 `qq_points(pvals)`。

评测命令：

```bash
python check.py 04-gc
```

### b7-int：逆正态变换

完成 `progress/<你>/work/b7-int/solution.py`，实现 `rank_int(x, offset=0.5)`。

评测命令：

```bash
python check.py 03-int
```

> 两个作业的函数签名与本课示例完全一致，可直接参照上方代码实现。
